# Phase 5 — Feature Engineering

**What:** create temporal, cyclical, lag, and rolling features.

**Why:** capture time-of-day, weekly, and recent-history information for next-hour PM2.5 forecasting.

**Leakage rule:** these features are deterministic and use only information at or before the feature timestamp. No statistical parameters are fitted here. Missing-value handling, outlier treatment, encoding, and scaling for modelling are fitted later **inside each training fold only**.

**Output:** `data/features/beijing_air_quality_features.csv`

## 1. Load target-integrated data

In [1]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd()
for candidate in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
    if (candidate / "src" / "project_config.py").is_file():
        PROJECT_ROOT = candidate
        break
sys.path.insert(0, str(PROJECT_ROOT))
from src.project_config import *

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

if not TARGET_DATA.exists():
    raise FileNotFoundError("Run 03_target_construction.ipynb first.")

df = pd.read_csv(TARGET_DATA, parse_dates=["datetime", "target_datetime"])
df = df.sort_values(["station", "datetime"]).reset_index(drop=True)

print("Rows:", f"{len(df):,}")
print("Columns:", len(df.columns))
print("Important: Phase 5 does not use globally fitted imputation, capping, or scaling parameters.")

Rows: 420,768
Columns: 22
Important: Phase 5 does not use globally fitted imputation, capping, or scaling parameters.


## 2. Temporal and cyclical features

**Decision:** represent repeating time patterns explicitly. `hour_sin` and `hour_cos` keep 23:00 close to 00:00; weekday/weekend and season provide calendar context.

In [2]:
df["day_of_week"] = df["datetime"].dt.dayofweek
df["is_weekend"] = df["day_of_week"].isin([5, 6]).astype(int)
df["season"] = df["month"].map({
    12: "Winter", 1: "Winter", 2: "Winter",
    3: "Spring", 4: "Spring", 5: "Spring",
    6: "Summer", 7: "Summer", 8: "Summer",
    9: "Autumn", 10: "Autumn", 11: "Autumn"
})
df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)

display(df[["datetime", "hour", "day_of_week", "is_weekend", "season", "hour_sin", "hour_cos"]].head(24))

,datetime,hour,day_of_week,is_weekend,season,hour_sin,hour_cos
0,2013-03-01 00:00:00,0,4,0,Spring,0.000000e+00,1.000000e+00
1,2013-03-01 01:00:00,1,4,0,Spring,2.588190e-01,9.659258e-01
2,2013-03-01 02:00:00,2,4,0,Spring,5.000000e-01,8.660254e-01
3,2013-03-01 03:00:00,3,4,0,Spring,7.071068e-01,7.071068e-01
4,2013-03-01 04:00:00,4,4,0,Spring,8.660254e-01,5.000000e-01
5,2013-03-01 05:00:00,5,4,0,Spring,9.659258e-01,2.588190e-01
6,2013-03-01 06:00:00,6,4,0,Spring,1.000000e+00,6.123234e-17
7,2013-03-01 07:00:00,7,4,0,Spring,9.659258e-01,-2.588190e-01
8,2013-03-01 08:00:00,8,4,0,Spring,8.660254e-01,-5.000000e-01
9,2013-03-01 09:00:00,9,4,0,Spring,7.071068e-01,-7.071068e-01


## 3. Leakage-safe lag and rolling features

**Decision:** calculate history separately within each station and continuous hourly segment. Lags use only earlier values; rolling statistics use `.shift(1)` so the current target-time value is not included.

In [3]:
work = df.copy()

gap = work.groupby("station")["datetime"].diff().ne(pd.Timedelta(hours=1))
work["_segment"] = gap.groupby(work["station"]).cumsum()
groups = work.groupby(["station", "_segment"], sort=False)

for lag in LAGS:
    work[f"PM2.5_lag_{lag}h"] = groups["PM2.5"].transform(
        lambda s: s.shift(lag)
    )

for window in ROLL_MEANS:
    work[f"PM2.5_roll_mean_{window}h"] = groups["PM2.5"].transform(
        lambda s: s.shift(1).rolling(window, min_periods=window).mean()
    )

for window in ROLL_STDS:
    work[f"PM2.5_roll_std_{window}h"] = groups["PM2.5"].transform(
        lambda s: s.shift(1).rolling(window, min_periods=window).std()
    )

# Validate lag timing exactly.
for lag in LAGS:
    lag_time = groups["datetime"].shift(lag)
    mask = work[f"PM2.5_lag_{lag}h"].notna()
    assert (work.loc[mask, "datetime"] - lag_time.loc[mask]).eq(pd.Timedelta(hours=lag)).all()

print("Validated lag features:", LAGS)
print("Validated rolling means:", ROLL_MEANS)
print("Validated rolling stds:", ROLL_STDS)

Validated lag features: [1, 2, 3, 6, 12, 24]
Validated rolling means: [3, 6, 24]
Validated rolling stds: [6, 24]


## 4. Feature screening

**Decision:** use correlation as evidence, not as automatic feature deletion. A feature may have a useful multivariate contribution even when its univariate correlation is modest.

In [4]:
feature_cols = (
    ["PM2.5"]
    + [f"PM2.5_lag_{x}h" for x in LAGS]
    + [f"PM2.5_roll_mean_{x}h" for x in ROLL_MEANS]
    + [f"PM2.5_roll_std_{x}h" for x in ROLL_STDS]
    + ["hour_sin", "hour_cos", "day_of_week", "is_weekend", "month", "season"]
)
numeric_for_corr = [
    c for c in feature_cols + [TARGET]
    if c in work.columns and pd.api.types.is_numeric_dtype(work[c])
]
screen = (
    work[numeric_for_corr]
    .corr()[TARGET]
    .drop(TARGET)
    .sort_values(key=lambda s: s.abs(), ascending=False)
    .rename("pearson_r")
    .to_frame()
)
screen["abs_r"] = screen["pearson_r"].abs()
screen["decision"] = np.where(
    screen["abs_r"] >= 0.5,
    "Strong/moderate candidate",
    "Retain initially; verify through model validation"
)
display(screen.round(3))

,pearson_r,abs_r,decision
PM2.5,0.969,0.969,Strong/moderate candidate
PM2.5_lag_1h,0.926,0.926,Strong/moderate candidate
PM2.5_roll_mean_3h,0.898,0.898,Strong/moderate candidate
PM2.5_lag_2h,0.884,0.884,Strong/moderate candidate
PM2.5_roll_mean_6h,0.858,0.858,Strong/moderate candidate
PM2.5_lag_3h,0.843,0.843,Strong/moderate candidate
PM2.5_lag_6h,0.733,0.733,Strong/moderate candidate
PM2.5_roll_mean_24h,0.701,0.701,Strong/moderate candidate
PM2.5_lag_12h,0.576,0.576,Strong/moderate candidate
PM2.5_lag_24h,0.392,0.392,Retain initially; verify through model validation


## 5. Save feature dataset

Rows are preserved here for auditability. Phase 6 removes only rows that cannot form a valid supervised example because the target or required lag/rolling history is unavailable.

In [5]:
FEATURE_DIR.mkdir(parents=True, exist_ok=True)

final_cols = [c for c in work.columns if c != "_segment"]
work[final_cols].to_csv(FEATURE_DATA, index=False)
screen.to_csv(FEATURE_DIR / "phase5_feature_target_correlation.csv")

print("Created:", FEATURE_DATA)
print("Rows:", f"{len(work):,}")
print("Columns:", len(final_cols))

Created: /Users/ruchi/Downloads/phase12_fixed/data/features/beijing_air_quality_features.csv
Rows: 420,768
Columns: 38
